# Dataset-agnostic SAS sanity checks

This notebook validates whether the SAS model behaves consistently on synthetic sentence pairs.

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import CrossEncoder

SAS_MODEL_NAME = "cross-encoder/stsb-roberta-large"
sas_model = CrossEncoder(SAS_MODEL_NAME)

def sas_similarity_score(text_a: str | None, text_b: str | None) -> float:
    if text_a is None or text_b is None:
        return np.nan

    a = str(text_a).strip()
    b = str(text_b).strip()
    if not a or not b:
        return np.nan

    try:
        score = float(sas_model.predict([(a, b)])[0])
        return score
    except Exception:
        return np.nan

In [ ]:
# --------------------------------
# Sanity check battery for SAS
# --------------------------------

sanity_cases = [
    {
        "case": "identical_sky",
        "text_a": "The sky is blue.",
        "text_b": "The sky is blue.",
        "reference_case": "paraphrase_car",
        "expected_relation": "higher",
    },
    {
        "case": "paraphrase_car",
        "text_a": "The car is fast.",
        "text_b": "The vehicle is quick.",
        "reference_case": "negation_sky",
        "expected_relation": "higher",
    },
    {
        "case": "negation_sky",
        "text_a": "The sky is blue.",
        "text_b": "The sky is not blue.",
        "reference_case": "paraphrase_car",
        "expected_relation": "lower",
    },
    {
        "case": "opposition_treatment",
        "text_a": "The treatment is effective.",
        "text_b": "The treatment is ineffective.",
        "reference_case": "paraphrase_car",
        "expected_relation": "lower",
    },
    {
        "case": "topic_opposed",
        "text_a": "Solar energy reduces long-term emissions.",
        "text_b": "Solar energy increases long-term emissions.",
        "reference_case": "paraphrase_car",
        "expected_relation": "lower",
    },
]

score_map = {}
for c in sanity_cases:
    score_map[c["case"]] = sas_similarity_score(c["text_a"], c["text_b"])

sanity_rows = []
for c in sanity_cases:
    case_name = c["case"]
    score = score_map.get(case_name, np.nan)
    ref_case = c["reference_case"]
    ref_score = score_map.get(ref_case, np.nan)
    relation = c["expected_relation"]

    passed = np.nan
    if pd.notna(score) and pd.notna(ref_score):
        if relation == "higher":
            passed = bool(score > ref_score)
        elif relation == "lower":
            passed = bool(score < ref_score)

    sanity_rows.append(
        {
            "case": case_name,
            "text_a": c["text_a"],
            "text_b": c["text_b"],
            "score_sas": score,
            "reference_case": ref_case,
            "reference_score": ref_score,
            "expected_relation": relation,
            "pass": passed,
        }
    )

# Explicit required check: score(paraphrase) > score(negation)
paraphrase_score = score_map.get("paraphrase_car", np.nan)
negation_score = score_map.get("negation_sky", np.nan)
explicit_pass = np.nan
if pd.notna(paraphrase_score) and pd.notna(negation_score):
    explicit_pass = bool(paraphrase_score > negation_score)

sanity_rows.append(
    {
        "case": "explicit_paraphrase_gt_negation",
        "text_a": "The car is fast.",
        "text_b": "The sky is not blue.",
        "score_sas": paraphrase_score,
        "reference_case": "negation_sky",
        "reference_score": negation_score,
        "expected_relation": "higher",
        "pass": explicit_pass,
    }
)

df_sanity = pd.DataFrame(sanity_rows)

pass_series = pd.to_numeric(df_sanity["pass"], errors="coerce")
sanity_pass_rate = float(pass_series.mean()) if pass_series.notna().any() else np.nan

from IPython.display import display
display(df_sanity)
print(f"Sanity pass rate: {sanity_pass_rate:.3f}" if pd.notna(sanity_pass_rate) else "Sanity pass rate: NaN")